In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [2]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device:{device}")

Using device:cuda


In [4]:
batch_size = 64
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_dataset=datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset=datasets.MNIST('./data', train=False, download=True, transform=transform)
train_loader=DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader=DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


100%|██████████| 9.91M/9.91M [00:16<00:00, 585kB/s] 
100%|██████████| 28.9k/28.9k [00:00<00:00, 139kB/s]
100%|██████████| 1.65M/1.65M [00:02<00:00, 633kB/s]
100%|██████████| 4.54k/4.54k [00:00<?, ?B/s]


In [5]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout2d(0.25)
        self.dropout2 = nn.Dropout2d(0.5)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc2(x)
        output = F.log_softmax(x, dim=1)
        return output

model=Net().to(device)
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(), lr=0.001)

In [6]:
def train(model, device, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % 100 == 0:
            print(f'Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)} ({100. * batch_idx / len(train_loader):.0f}%)]\tLoss: {loss.item():.6f}')

def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item()  # sum up batch loss
            pred = output.argmax(dim=1, keepdim=True)  # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)

    print(f'\nTest set: Average loss: {test_loss:.4f}, Accuracy: {correct}/{len(test_loader.dataset)} ({100. * correct / len(test_loader.dataset):.0f}%)\n')

In [8]:
epochs = 10
if __name__ == '__main__':
    for epoch in range(epochs):
        train(model, device, train_loader, optimizer, epoch)
        test(model, device, test_loader)

c:\Users\shimmer\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\functional.py:1535: UserWarning: dropout2d: Received a 2-D input to dropout2d, which is deprecated and will result in an error in a future release. To retain the behavior and silence this warning, please use dropout instead. Note that dropout2d exists to provide channel-wise dropout on inputs with 2 spatial dimensions, a channel dimension, and an optional batch dimension (i.e. 3D or 4D inputs).
  warnings.warn(warn_msg)


Train Epoch: 0 [0/60000 (0%)]	Loss: 2.303843
Train Epoch: 0 [6400/60000 (11%)]	Loss: 0.209370
Train Epoch: 0 [12800/60000 (21%)]	Loss: 0.123673
Train Epoch: 0 [19200/60000 (32%)]	Loss: 0.116015
Train Epoch: 0 [25600/60000 (43%)]	Loss: 0.284251
Train Epoch: 0 [32000/60000 (53%)]	Loss: 0.190178
Train Epoch: 0 [38400/60000 (64%)]	Loss: 0.082731
Train Epoch: 0 [44800/60000 (75%)]	Loss: 0.160887
Train Epoch: 0 [51200/60000 (85%)]	Loss: 0.216719
Train Epoch: 0 [57600/60000 (96%)]	Loss: 0.078953

Test set: Average loss: 0.0008, Accuracy: 9845/10000 (98%)

Train Epoch: 1 [0/60000 (0%)]	Loss: 0.074714
Train Epoch: 1 [6400/60000 (11%)]	Loss: 0.065968
Train Epoch: 1 [12800/60000 (21%)]	Loss: 0.051203
Train Epoch: 1 [19200/60000 (32%)]	Loss: 0.151775
Train Epoch: 1 [25600/60000 (43%)]	Loss: 0.131680
Train Epoch: 1 [32000/60000 (53%)]	Loss: 0.014445
Train Epoch: 1 [38400/60000 (64%)]	Loss: 0.025505
Train Epoch: 1 [44800/60000 (75%)]	Loss: 0.039024
Train Epoch: 1 [51200/60000 (85%)]	Loss: 0.124881
T